In [ ]:
!pip install pydantic==1.10.12

In [ ]:
!pip install ydata-profiling

In [ ]:
from ydata_profiling import ProfileReport
import pandas as pd
import numpy as np
import plotly.express as px
from matplotlib import pyplot as plt
import seaborn as sns

In [ ]:
data = pd.read_csv("final test1.csv")

In [ ]:
data.describe(include='all')

In [ ]:
profile = ProfileReport(data)
profile

In [ ]:
data

In [ ]:
data.groupby('Category').size().plot(kind='barh', color=sns.palettes.mpl_palette('Dark2'))
plt.gca().spines[['top', 'right',]].set_visible(False)

In [ ]:
# Create the treemap
fig = px.treemap(
    data,
    path=["Marketing/Advertisement"],       # Define hierarchy (only one level here)
    values="Rating", # Numeric values for the size
    title="Treemap of Ratings by Marketing Channel",
    color="Rating",  # Color by subscription count
    color_continuous_scale="Blues"  # Color scheme
)

# Show the treemap
fig.show()

In [ ]:
!pip install autoviz

In [ ]:
from autoviz.AutoViz_Class import AutoViz_Class
# creating an Autoviz instance
AV = AutoViz_Class()

# generating data visualization automatically
AV.AutoViz(
    filename='',
    sep=',',
    depVar='',
    dfte=data,
    header=0,
    verbose=0,
    lowess=False,
    chart_format='svg',
    max_rows_analyzed=10000,
    max_cols_analyzed=30
)

#Data Preprocessing

In [ ]:
data.info()

In [ ]:
# Convert date columns with the correct format
data['Order Date'] = pd.to_datetime(data['Order Date'], format='%d/%m/%Y', errors='coerce')
data['Delivery Date'] = pd.to_datetime(data['Delivery Date'], format='%d/%m/%Y', errors='coerce')
data['Cancellation Date'] = pd.to_datetime(data['Cancellation Date'], format='%d/%m/%Y', errors='coerce')


In [ ]:
# Replace "Cancelled" with NaN
data['Shipping Charges'] = pd.to_numeric(data['Shipping Charges'], errors='coerce')

data['Shipping Charges'].fillna(0, inplace=True)


In [ ]:
data.info()

In [ ]:
data.isnull().sum()

In [ ]:
null_cancellation_date_data = data[data['Cancellation Date'].isnull()]

# Count the occurrence of each RESULT value
payment_status_counts = null_cancellation_date_data['Payment Status'].value_counts()
print("\nCounts of Payment Status values where Cancellation Date is null:")
display(payment_status_counts)

In [ ]:
duplicates = data[data.duplicated()]
print('Duplicate rows: ', len(duplicates))

In [ ]:
data.nunique()

In [ ]:
# Engineer new features
data['Discount Percentage'] = (data['MRP'] - data['Discount Price']) / data['MRP']
data['Profit Margin'] = data['MRP'] - data['Discount Price']
data['Avg Time Per Click'] = data['Time Spent on Website'] / data['No of Clicks']
data['Is Repeat Customer'] = data['Customer ID'].duplicated(keep=False).astype(int)

# Handle any infinite or NaN values created during feature engineering
data.replace([float('inf'), -float('inf')], None, inplace=True)
data.fillna(0, inplace=True)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Convert 'Total Order Value' to numeric, coercing errors
data['Total Order Value'] = pd.to_numeric(data['Total Order Value'], errors='coerce')
# Fill NaN values that resulted from coercion with 0
data['Total Order Value'].fillna(0, inplace=True)

# Split data into train, validation, and test sets
train, temp = train_test_split(data, test_size=0.4, random_state=42)
val, test = train_test_split(temp, test_size=0.5, random_state=42)

# Define categorical and numerical columns
categorical_cols = [
    'Gender', 'Category', 'Subscription', 'Marketing/Advertisement',
    'Ship Mode', 'Order Status', 'Payment Method', 'Payment Status'
]
numerical_cols = [
    'Customer Age', 'MRP', 'Discount Price', 'Time Spent on Website', 'Rating',
    'No of Clicks', 'Shipping Charges', 'Total Order Value'
]

# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('num', StandardScaler(), numerical_cols)
    ],
    remainder='passthrough'
)

# Fit the preprocessor on the training data
preprocessor.fit(train)

# Transform train, val, and test sets
train_processed = preprocessor.transform(train)
val_processed = preprocessor.transform(val)
test_processed = preprocessor.transform(test)

# Convert NumPy array back to DataFrame
train_processed_df = pd.DataFrame(
    train_processed,
    columns=preprocessor.get_feature_names_out()
)
val_processed_df = pd.DataFrame(
    val_processed,
    columns=preprocessor.get_feature_names_out()
)
test_processed_df = pd.DataFrame(
    test_processed,
    columns=preprocessor.get_feature_names_out()
)

print("Train set processed:")
train_processed_df.head()

In [ ]:
train_processed_df.info()

In [ ]:
# List of columns to convert from object to numeric
columns_to_convert = [
    'cat__Gender_Female',
    'cat__Gender_Male',
    'num__MRP',
    'num__Discount Price',
    'cat__Category_Branded',
    'cat__Category_Local',
    'cat__Category_Imported'
]

# Convert specified columns to numeric, coercing errors
for column in columns_to_convert:
    train_processed_df[column] = pd.to_numeric(train_processed_df[column], errors='coerce')
    test_processed_df[column] = pd.to_numeric(test_processed_df[column], errors='coerce')

print(train_processed_df.dtypes)

#Modelling

In [ ]:
!pip install catboost

In [ ]:
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from catboost import CatBoostRegressor

In [ ]:
# Select the columns from train_processed_df that are numerical after preprocessing
# These are the 'cat__' and 'num__' prefixed columns, excluding 'num__Total Order Value' which is the target.
features_from_preprocessed_df = [col for col in train_processed_df.columns if (col.startswith('cat__') or col.startswith('num__')) and col != 'num__Total Order Value']
X_train_transformed_features = train_processed_df[features_from_preprocessed_df]
y_train = pd.to_numeric(train_processed_df['num__Total Order Value'], errors='coerce') # Ensure y_train is numeric

# Select the additional engineered features from train
# These engineered features were created on the 'data' DataFrame and are numeric.
engineered_features_cols = ['Discount Percentage', 'Profit Margin', 'Avg Time Per Click', 'Is Repeat Customer']
X_train_engineered_features = train[engineered_features_cols].reset_index(drop=True)

# Concatenate all features for X_train
X_train = pd.concat([X_train_transformed_features, X_train_engineered_features], axis=1)

# Repeat for the test set
features_from_preprocessed_df_test = [col for col in test_processed_df.columns if (col.startswith('cat__') or col.startswith('num__')) and col != 'num__Total Order Value']
X_test_transformed_features = test_processed_df[features_from_preprocessed_df_test]
y_test = pd.to_numeric(test_processed_df['num__Total Order Value'], errors='coerce') # Ensure y_test is numeric

# Select the additional engineered features from test
X_test_engineered_features = test[engineered_features_cols].reset_index(drop=True)

# Concatenate all features for X_test
X_test = pd.concat([X_test_transformed_features, X_test_engineered_features], axis=1)

# Ensure all columns in X_train and X_test are numeric and fill any potential NaNs
X_train = X_train.apply(pd.to_numeric, errors='coerce')
X_test = X_test.apply(pd.to_numeric, errors='coerce')
y_train.fillna(0, inplace=True)
y_test.fillna(0, inplace=True)
X_train.fillna(0, inplace=True)
X_test.fillna(0, inplace=True)


#Decision Tree

In [ ]:
# Decision Tree Regressor
dt_model = DecisionTreeRegressor(random_state=42)
dt_model.fit(X_train, y_train)
dt_predictions = dt_model.predict(X_test)
dt_mse = mean_squared_error(y_test, dt_predictions)
dt_rmse = np.sqrt(dt_mse)
dt_mae = mean_absolute_error(y_test, dt_predictions)
dt_r2 = r2_score(y_test, dt_predictions)
print(f"Decision Tree MSE: {dt_mse}")
print(f"Decision Tree RMSE: {dt_rmse}")
print(f"Decision Tree MAE: {dt_mae}")
print(f"Decision Tree R-squared: {dt_r2}")

In [ ]:
# Decision Tree Regressor - Hyperparameter Tuning
dt_param_grid = {
    'max_depth': [5, 10, 20, None],
    'min_samples_split': [2, 5, 10]
}
dt_grid_search = GridSearchCV(DecisionTreeRegressor(random_state=42), dt_param_grid, cv=5, scoring='neg_mean_squared_error')
dt_grid_search.fit(X_train, y_train)
dt_best_model = dt_grid_search.best_estimator_
dt_predictions = dt_best_model.predict(X_test)
dt_mse = mean_squared_error(y_test, dt_predictions)
dt_rmse = np.sqrt(dt_mse)
dt_mae = mean_absolute_error(y_test, dt_predictions)
dt_r2 = r2_score(y_test, dt_predictions)

print(f"Decision Tree Best Params: {dt_grid_search.best_params_}")
print(f"Decision Tree MSE: {dt_mse}")
print(f"Decision Tree RMSE: {dt_rmse}")
print(f"Decision Tree MAE: {dt_mae}")
print(f"Decision Tree R-squared: {dt_r2}")

#Random Forest

In [ ]:
# Random Forest Regressor
rf_model = RandomForestRegressor(random_state=42)
rf_model.fit(X_train, y_train)
rf_predictions = rf_model.predict(X_test)
rf_mse = mean_squared_error(y_test, rf_predictions)
rf_rmse = np.sqrt(rf_mse)
rf_mae = mean_absolute_error(y_test, rf_predictions)
rf_r2 = r2_score(y_test, rf_predictions)

print(f"Random Forest MSE: {rf_mse}")
print(f"Random Forest RMSE: {rf_rmse}")
print(f"Random Forest MAE: {rf_mae}")
print(f"Random Forest R-squared: {rf_r2}")

In [ ]:
# Random Forest Regressor - Hyperparameter Tuning
rf_param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15]
}
rf_grid_search = GridSearchCV(RandomForestRegressor(random_state=42), rf_param_grid, cv=5, scoring='neg_mean_squared_error')
rf_grid_search.fit(X_train, y_train)
rf_best_model = rf_grid_search.best_estimator_
rf_predictions = rf_best_model.predict(X_test)
rf_mse = mean_squared_error(y_test, rf_predictions)
rf_rmse = np.sqrt(rf_mse)
rf_mae = mean_absolute_error(y_test, rf_predictions)
rf_r2 = r2_score(y_test, rf_predictions)

print(f"Random Forest Best Params: {rf_grid_search.best_params_}")
print(f"Random Forest MSE: {rf_mse}")
print(f"Random Forest RMSE: {rf_rmse}")
print(f"Random Forest MAE: {rf_mae}")
print(f"Random Forest R-squared: {rf_r2}")

#Neural Network

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.backend import clear_session

In [ ]:
# Convert targets to numpy arrays of type float
X_train = np.array(X_train, dtype=np.float32)
X_test = np.array(X_test, dtype=np.float32)
y_train = np.array(y_train, dtype=np.float32)
y_test = np.array(y_test, dtype=np.float32)

In [ ]:
# Clear any previous session
clear_session()

In [ ]:
# Build Neural Network model
nn_model = Sequential([
    Dense(32, input_dim=X_train.shape[1], activation='relu'),
    Dense(8, activation='relu'),
    Dense(1)  # Output layer for regression
])

In [ ]:
# Compile the model
nn_model.compile(optimizer=Adam(learning_rate=0.001), loss='mse')

In [ ]:
# Train the model
history = nn_model.fit(X_train, y_train, epochs=20, batch_size=32, validation_split=0.2, verbose=1)

In [ ]:
# Make predictions
nn_predictions = nn_model.predict(X_test)

# Calculate evaluation metrics
nn_mse = mean_squared_error(y_test, nn_predictions)
nn_rmse = np.sqrt(nn_mse)
nn_mae = mean_absolute_error(y_test, nn_predictions)
nn_r2 = r2_score(y_test, nn_predictions)

# Print the results
print(f"Neural Network MSE: {nn_mse}")
print(f"Neural Network RMSE: {nn_rmse}")
print(f"Neural Network MAE: {nn_mae}")
print(f"Neural Network R-squared: {nn_r2}")

In [ ]:
!pip install keras-tuner

In [ ]:
# Neural Network - Hyperparameter Tuning

from keras_tuner import RandomSearch

# Convert to NumPy arrays with appropriate type
X_train = np.array(X_train, dtype=np.float32)
y_train = np.array(y_train, dtype=np.float32)
X_test = np.array(X_test, dtype=np.float32)
y_test = np.array(y_test, dtype=np.float32)

# Clear any previous session
clear_session()

# Define the model-building function
def build_model(hp):
    model = Sequential()
    model.add(Dense(units=hp.Int('units_layer1', min_value=16, max_value=128, step=16),
                    input_dim=X_train.shape[1], activation='relu'))
    model.add(Dense(units=hp.Int('units_layer2', min_value=4, max_value=64, step=8),
                    activation='relu'))
    model.add(Dense(1))
    model.compile(optimizer=Adam(learning_rate=hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])),
                  loss='mse')
    return model

# Initialize the Keras Tuner
tuner = RandomSearch(
    build_model,
    objective='val_loss',
    max_trials=10,  # Number of hyperparameter combinations to try
    executions_per_trial=1,  # Number of models to train for each combination
    directory='my_tuner_dir',
    project_name='nn_hyperparameter_tuning'
)

# Run the hyperparameter search
tuner.search(X_train, y_train, epochs=20, batch_size=32, validation_split=0.2, verbose=1)

# Retrieve and train the best model
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

# Build and train the best model
best_model = tuner.hypermodel.build(best_hps)
history = best_model.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.2, verbose=1)

# Evaluate the best model
nn_loss = best_model.evaluate(X_test, y_test, verbose=0)
print(f"Best Neural Network Loss (MSE): {nn_loss}")

# Make predictions with the best model
nn_predictions = best_model.predict(X_test)

# Calculate the evaluation metrics
nn_mse = mean_squared_error(y_test, nn_predictions)
nn_rmse = np.sqrt(nn_mse)
nn_mae = mean_absolute_error(y_test, nn_predictions)
nn_r2 = r2_score(y_test, nn_predictions)

# Print the results
print(f"Best Neural Network Loss (MSE): {nn_loss}")
print(f"Best Neural Network MSE: {nn_mse}")
print(f"Best Neural Network RMSE: {nn_rmse}")
print(f"Best Neural Network MAE: {nn_mae}")
print(f"Best Neural Network R-squared: {nn_r2}")

In [ ]:
# Print the best hyperparameters
print("Best Hyperparameters:")
print(f"units_layer1: {best_hps.get('units_layer1')}")
print(f"units_layer2: {best_hps.get('units_layer2')}")
print(f"learning_rate: {best_hps.get('learning_rate')}")

In [ ]:
# Clear any previous session
clear_session()

#CatBoost

In [ ]:
# CatBoost model
cat_model = CatBoostRegressor(random_state=42, verbose=0)
cat_model.fit(X_train, y_train)
cat_predictions = cat_model.predict(X_test)

cat_mse = mean_squared_error(y_test, cat_predictions)
cat_rmse = np.sqrt(cat_mse)
cat_mae = mean_absolute_error(y_test, cat_predictions)
cat_r2 = r2_score(y_test, cat_predictions)

# Print the results
print(f"CatBoost MSE: {cat_mse}")
print(f"CatBoost RMSE: {cat_rmse}")
print(f"CatBoost MAE: {cat_mae}")
print(f"CatBoost R-squared: {cat_r2}")

In [ ]:
# CatBoost - hyperparameter tuning
param_grid = {
    'iterations': [100, 200],             # Number of boosting iterations
    'depth': [3, 5, 7],                   # Depth of the trees
    'learning_rate': [0.01, 0.1],         # Learning rate
    'l2_leaf_reg': [1, 3, 5]              # L2 regularization coefficient
}

# Set up GridSearchCV
grid_search = GridSearchCV(estimator=cat_model,
                           param_grid=param_grid,
                           scoring='neg_mean_squared_error',
                           cv=5,
                           verbose=1,
                           n_jobs=-1)
grid_search.fit(X_train, y_train)

# Get the best parameters and best score
best_params = grid_search.best_params_
best_score = -grid_search.best_score_

print(f"Best Parameters: {best_params}")
# print(f"Best Cross-Validated MSE: {best_score}")

best_model = grid_search.best_estimator_
cat_predictions = best_model.predict(X_test)

# Calculate evaluation metrics
cat_mse = mean_squared_error(y_test, cat_predictions)
cat_rmse = np.sqrt(cat_mse)
cat_mae = mean_absolute_error(y_test, cat_predictions)
cat_r2 = r2_score(y_test, cat_predictions)

# Print the results
print(f"CatBoost MSE: {cat_mse}")
print(f"CatBoost RMSE: {cat_rmse}")
print(f"CatBoost MAE: {cat_mae}")
print(f"CatBoost R-squared: {cat_r2}")

#H2O Gradient Boosting Machine (GBM)

In [ ]:
!pip install h2o

In [ ]:
import h2o
from h2o.grid.grid_search import H2OGridSearch
from h2o.estimators.gbm import H2OGradientBoostingEstimator

# Initialize H2O
h2o.init()

# Define target column early
target = 'num__Total Order Value'

# Get the full list of predictor column names (as constructed in cell xG0ZSStIhV5T)
all_predictor_cols = features_from_preprocessed_df + engineered_features_cols

# Convert numpy arrays back to pandas DataFrames/Series for H2O
X_train_h2o_df = pd.DataFrame(X_train, columns=all_predictor_cols)
y_train_h2o_series = pd.Series(y_train, name=target)

X_test_h2o_df = pd.DataFrame(X_test, columns=all_predictor_cols)
y_test_h2o_series = pd.Series(y_test, name=target)

# Convert pandas DataFrames to H2O Frames
h2o_train = h2o.H2OFrame(pd.concat([X_train_h2o_df, y_train_h2o_series], axis=1))
h2o_test = h2o.H2OFrame(pd.concat([X_test_h2o_df, y_test_h2o_series], axis=1))

# Define predictors
predictors = all_predictor_cols # Use the correctly derived list of predictor columns

# Split the H2O train dataset into training and validation sets
train, valid = h2o_train.split_frame(ratios=[0.8], seed=42)

# Define hyperparameter grid for H2O GBM
hyper_params = {
    'max_depth': [3, 5, 7, 9],
    'ntrees': [50, 100, 150],
    'learn_rate': [0.01, 0.1, 0.2],
    'sample_rate': [0.8, 1.0],
    'col_sample_rate': [0.8, 1.0]
}

# Initialize H2O GBM
gbm = H2OGradientBoostingEstimator(
    seed=42,               # Ensure reproducibility
    balance_classes=True,  # Handle imbalanced datasets
    stopping_rounds=5,     # Early stopping for better performance
    stopping_metric="RMSE" # Metric for early stopping
)

# Perform grid search
grid_search = H2OGridSearch(
    model=gbm,
    hyper_params=hyper_params,
    search_criteria={"strategy": "Cartesian"}  # Try all combinations
)

# Train grid search models
grid_search.train(x=predictors, y=target, training_frame=train, validation_frame=valid)

# View grid search results sorted by RMSE
grid_results = grid_search.get_grid(sort_by="rmse", decreasing=False)
print(grid_results)

# Get the best model
best_model = grid_results.models[0]
print("Best Model Performance:", best_model.model_performance(valid))
# Print the best hyperparameters
print("Best Hyperparameters:", best_model.params)

# Predict on the test set
predictions = best_model.predict(h2o_test)

# Print predictions
print(predictions)

# Shutdown H2O when done
h2o.shutdown(prompt=False)